# V18 BCR/TCR Data Deep Scan + h5ad Analysis
# V18 BCR/TCR — Corrected Folder Scan + Full Analysis
**Fix:** Correct paths for gut_2021 supplementary and extracted files
**Date:** 2026-03-14
**Purpose:**
1. Scan Zhang Gut supplementary folder and extracted files folder
2. Analyze h5ad BCR/TCR column structure in detail
3. Produce tissue-separated × stage × donor summary

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.9 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [11]:
# Cell 1: Mount Drive & scan both folders
from google.colab import drive
drive.mount('/content/drive')
import os, glob
import pandas as pd
import numpy as np

# --- Folder 1: Gut supplementary data ---
PATHS = {
    'gut_2021 (docs)': '/content/drive/MyDrive/ITLAS/docs/gut_2021',
    'extracted (data)': '/content/drive/MyDrive/ITLAS/data/extracted',
}

all_found_files = []

for label, path in PATHS.items():
    print(f'\n{"="*70}')
    print(f'SCANNING: {label}')
    print(f'Path: {path}')
    print(f'{"="*70}')
    if not os.path.exists(path):
        print(f'  ⚠️ Path does not exist!')
        continue
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, '').count(os.sep)
        if depth > 3:
            continue
        indent = '  ' * (depth + 1)
        rel = os.path.relpath(root, path)
        if rel != '.':
            print(f'{indent}📁 {os.path.basename(root)}/')
        for f in sorted(files):
            fpath = os.path.join(root, f)
            fsize = os.path.getsize(fpath)
            if fsize > 1024*1024*1024:
                sz_str = f'{fsize/1024/1024/1024:.1f} GB'
            elif fsize > 1024*1024:
                sz_str = f'{fsize/1024/1024:.1f} MB'
            else:
                sz_str = f'{fsize/1024:.1f} KB'
            fl = f.lower()
            ext = os.path.splitext(f)[1].lower()
            flag = ''
            if any(kw in fl for kw in ['bcr','tcr','vdj','clone','contig','repertoire']):
                flag = ' ⭐ BCR/TCR'
            elif ext in ['.csv','.tsv','.txt','.xlsx']:
                flag = ' 📊'
            print(f'{indent}  {f} ({sz_str}){flag}')
            all_found_files.append({'path': fpath, 'name': f, 'size': fsize, 'ext': ext, 'source': label})

print(f'\n{"="*70}')
print(f'Total files found: {len(all_found_files)}')
bcr_tcr_files = [f for f in all_found_files if any(kw in f["name"].lower() for kw in ['bcr','tcr','vdj','clone','contig'])]
print(f'BCR/TCR related files: {len(bcr_tcr_files)}')
csv_files = [f for f in all_found_files if f['ext'] in ['.csv','.tsv','.txt']]
print(f'CSV/TSV/TXT files: {len(csv_files)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

SCANNING: gut_2021 (docs)
Path: /content/drive/MyDrive/ITLAS/docs/gut_2021
    gutjnl-2021-325915.pdf (9.3 MB)
    gutjnl-2021-325915supp001_data_supplement.pdf (10.1 MB)
    gutjnl-2021-325915supp002_data_supplement.xlsx (16.4 KB) 📊
    gutjnl-2021-325915supp003_data_supplement.xlsx (23.0 KB) 📊
    gutjnl-2021-325915supp004_data_supplement.xlsx (1.6 MB) 📊
    gutjnl-2021-325915supp005_data_supplement.xlsx (17.6 KB) 📊
    gutjnl-2021-325915supp006_data_supplement.xlsx (10.7 MB) 📊
    gutjnl-2021-325915supp007_data_supplement.xlsx (37.3 KB) 📊
    gutjnl-2021-325915supp008_data_supplement.xlsx (12.2 KB) 📊
    gutjnl-2021-325915supp009_data_supplement.xlsx (12.1 KB) 📊
    gutjnl-2021-January-72-1-153-F1.large.jpg (234.2 KB)
    gutjnl-2021-January-72-1-153-F2.large.jpg (199.2 KB)
    gutjnl-2021-January-72-1-153-F3.large.jpg (178.0 KB)
    gutjnl-2021-January-7

In [12]:
# Cell 2: Preview ALL CSV/TSV files — identify BCR/TCR data
print(f'\n{"="*70}')
print('PREVIEWING ALL CSV/TSV FILES')
print(f'{"="*70}')

for finfo in csv_files:
    fpath = finfo['path']
    fname = finfo['name']
    sz = finfo['size']
    sz_str = f'{sz/1024:.1f} KB' if sz < 1024*1024 else f'{sz/1024/1024:.1f} MB'
    print(f'\n{"─"*60}')
    print(f'📄 {fname} ({sz_str}) — from {finfo["source"]}')
    try:
        # Auto-detect separator
        with open(fpath, 'r') as fh:
            first_line = fh.readline()
        sep = '\t' if '\t' in first_line else ','
        df = pd.read_csv(fpath, sep=sep, nrows=5, low_memory=False)
        print(f'   Columns ({len(df.columns)}): {list(df.columns[:15])}', end='')
        if len(df.columns) > 15:
            print(f'... +{len(df.columns)-15} more')
        else:
            print()
        # Full row count
        if sz < 100*1024*1024:  # only for files < 100MB
            n_rows = sum(1 for _ in open(fpath)) - 1
            print(f'   Rows: {n_rows:,}')
        # Check BCR/TCR relevance
        cols_str = ' '.join(c.lower() for c in df.columns)
        is_vdj = any(kw in cols_str for kw in
                     ['bcr','tcr','vdj','clone','cdr3','v_gene','j_gene',
                      'chain','contig','barcode','igh','igk','igl','tra','trb',
                      'clonotype','productive'])
        if is_vdj:
            print(f'   ⭐⭐ BCR/TCR RELATED! ⭐⭐')
            print(f'   First 2 rows:')
            df2 = pd.read_csv(fpath, sep=sep, nrows=2, low_memory=False)
            print(df2.to_string())
    except Exception as e:
        print(f'   Error: {e}')


PREVIEWING ALL CSV/TSV FILES

────────────────────────────────────────────────────────────
📄 GSM5519467_P190604_Blood_1.txt (128.2 MB) — from extracted (data)
   Columns (1): ['AAACCTGAGCTGAACG-1-P190604_Blood_1 "AAACCTGAGGGTGTTG-1-P190604_Blood_1" "AAACCTGGTTCGGGCT-1-P190604_Blood_1" "AAACCTGTCCCAAGTA-1-P190604_Blood_1" "AAACGGGCACGCTTTC-1-P190604_Blood_1" "AAACGGGTCAACACTG-1-P190604_Blood_1" "AAACGGGTCGCCATAA-1-P190604_Blood_1" "AAAGATGCACTTAACG-1-P190604_Blood_1" "AAAGATGGTTCTGAAC-1-P190604_Blood_1" "AAAGCAATCCATGCTC-1-P190604_Blood_1" "AAAGTAGAGATACACA-1-P190604_Blood_1" "AAAGTAGCAGGCAGTA-1-P190604_Blood_1" "AAAGTAGTCTCCGGTT-1-P190604_Blood_1" "AAAGTAGTCTGTCCGT-1-P190604_Blood_1" "AAATGCCAGCGTCAAG-1-P190604_Blood_1" "AAATGCCCAATCGAAA-1-P190604_Blood_1" "AAATGCCCATGGATGG-1-P190604_Blood_1" "AAATGCCGTCGACTGC-1-P190604_Blood_1" "AAATGCCTCAAGGTAA-1-P190604_Blood_1" "AACACGTAGTACACCT-1-P190604_Blood_1" "AACACGTCACAGGAGT-1-P190604_Blood_1" "AACACGTTCCTTTCGG-1-P190604_Blood_1" "AACCATGAG

In [13]:
# Cell 3: h5ad BCR deep analysis — Stage × Tissue × Donor
import scanpy as sc

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
adata = sc.read_h5ad(DATA_PATH, backed='r')
obs = adata.obs.copy()
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

print(f'{"="*70}')
print('BCR DEEP ANALYSIS: Stage × Tissue')
print(f'{"="*70}')

# BCR by Stage × Tissue
bcr_xt = obs.groupby(['Stage', 'tissue'], observed=True).agg(
    total=('BCR_clone.id', 'size'),
    bcr=('BCR_clone.id', lambda x: x.notna().sum()),
    unique_clones=('BCR_clone.id', lambda x: x.dropna().nunique()),
).reset_index()
bcr_xt['pct'] = (bcr_xt['bcr'] / bcr_xt['total'] * 100).round(1)
bcr_xt['pct_singleton'] = ((bcr_xt['unique_clones'] / bcr_xt['bcr']) * 100).round(1)
print('\n--- BCR cells by Stage × Tissue ---')
for stage in ['NL','IT','IA','AR','CR']:
    rows = bcr_xt[bcr_xt['Stage']==stage]
    for _, r in rows.iterrows():
        print(f'  {stage}/{r.tissue}: {int(r.bcr):,}/{int(r.total):,} ({r.pct}%), '
              f'unique={int(r.unique_clones):,}, singleton~{r.pct_singleton}%')

# Isotype by Stage × Tissue
print(f'\n--- Isotype % by Stage × Tissue ---')
for tissue_val in ['Liver', 'Blood']:
    print(f'\n  {tissue_val}:')
    for stage in ['NL','IT','IA','AR','CR']:
        sub = obs[(obs['Stage']==stage) & (obs['tissue']==tissue_val) & obs['BCR_CType'].notna()]
        if len(sub) == 0:
            print(f'    {stage}: no data')
            continue
        iso = sub['BCR_CType'].value_counts()
        total = iso.sum()
        parts = [f'{k}={v/total*100:.1f}%' for k, v in iso.items()]
        print(f'    {stage} (n={total}): {" | ".join(parts)}')

BCR DEEP ANALYSIS: Stage × Tissue

--- BCR cells by Stage × Tissue ---
  NL/Blood: 1,817/18,360 (9.9%), unique=1,817, singleton~100.0%
  NL/Liver: 1,164/24,219 (4.8%), unique=1,077, singleton~92.5%
  IT/Blood: 3,375/29,678 (11.4%), unique=3,354, singleton~99.4%
  IT/Liver: 610/19,501 (3.1%), unique=599, singleton~98.2%
  IA/Blood: 3,803/30,256 (12.6%), unique=3,652, singleton~96.0%
  IA/Liver: 1,389/32,289 (4.3%), unique=1,366, singleton~98.3%
  AR/Blood: 547/27,706 (2.0%), unique=546, singleton~99.8%
  AR/Liver: 59/17,746 (0.3%), unique=59, singleton~100.0%
  CR/Blood: 0/30,408 (0.0%), unique=0, singleton~nan%
  CR/Liver: 0/12,837 (0.0%), unique=0, singleton~nan%

--- Isotype % by Stage × Tissue ---

  Liver:
    NL (n=1161): IGHG=38.8% | IGHM=37.0% | IGHA=15.0% | IGHD=9.3%
    IT (n=609): IGHM=47.8% | IGHG=34.3% | IGHA=14.6% | IGHD=3.3%
    IA (n=1388): IGHM=57.6% | IGHG=22.3% | IGHA=16.1% | IGHD=4.1%
    AR (n=59): IGHM=50.8% | IGHA=25.4% | IGHG=22.0% | IGHD=1.7%
    CR: no data

  

In [14]:
# Cell 4: DONOR-LEVEL BCR metrics (Liver & Blood separately)
from scipy.stats import mannwhitneyu

print(f'{"="*70}')
print('DONOR-LEVEL BCR METRICS + Mann-Whitney NL→IT')
print(f'{"="*70}')

def donor_bcr(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        bcr = grp[grp['BCR_clone.id'].notna()]
        nb = len(bcr)
        if nb == 0:
            rows.append({'Stage':stage,'donor':donor,'n_bcr':0,'pct_bcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'pct_IgM':np.nan,'pct_IgG':np.nan,'pct_IgA':np.nan,
                         'pct_switched':np.nan,'top_clone_size':0})
            continue
        cc = bcr['BCR_clone.id'].value_counts()
        nu = len(cc)
        ns = (cc==1).sum()
        if nu > 1:
            fr = cc.values / cc.sum()
            ent = -np.sum(fr * np.log2(fr))
            clon = 1 - ent/np.log2(nu)
        else:
            clon = 0
        iso = bcr['BCR_CType'].value_counts()
        it = iso.sum()
        igm = iso.get('IGHM',0)/it*100 if it>0 else np.nan
        igg = iso.get('IGHG',0)/it*100 if it>0 else np.nan
        iga = iso.get('IGHA',0)/it*100 if it>0 else np.nan
        switched = (igg if igg else 0) + (iga if iga else 0)
        rows.append({'Stage':stage,'donor':donor,'n_bcr':nb,
                     'pct_bcr':nb/n*100,'clonality':clon,
                     'pct_singleton':ns/nu*100 if nu>0 else np.nan,
                     'pct_IgM':igm,'pct_IgG':igg,'pct_IgA':iga,
                     'pct_switched':switched,'top_clone_size':cc.max()})
    return pd.DataFrame(rows)

for tissue_val in ['Liver','Blood']:
    df = donor_bcr(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — Donor-level BCR')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[df['Stage']==stage]
        if len(s)==0 or s['n_bcr'].sum()==0:
            print(f'  {stage}: no BCR data')
            continue
        valid = s[s['n_bcr']>0]
        print(f'  {stage} ({len(valid)} donors with BCR):')
        print(f'    BCR cells: {valid.n_bcr.mean():.0f}±{valid.n_bcr.std():.0f}')
        print(f'    Clonality: {valid.clonality.mean():.4f}±{valid.clonality.std():.4f}')
        print(f'    Singleton: {valid.pct_singleton.mean():.1f}%')
        print(f'    IgM: {valid.pct_IgM.mean():.1f}% | IgG: {valid.pct_IgG.mean():.1f}% | IgA: {valid.pct_IgA.mean():.1f}%')
        print(f'    Class-switched: {valid.pct_switched.mean():.1f}% | Top clone: {valid.top_clone_size.mean():.1f}')

    # Mann-Whitney: NL vs IT
    nl = df[(df['Stage']=='NL') & (df['n_bcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_bcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  --- Mann-Whitney NL→IT ({tissue_val}) ---')
        for metric in ['clonality','pct_singleton','pct_IgM','pct_IgG','pct_IgA','pct_switched','pct_bcr']:
            nl_vals = nl[metric].dropna()
            it_vals = it[metric].dropna()
            if len(nl_vals)>=2 and len(it_vals)>=2:
                stat, p = mannwhitneyu(nl_vals, it_vals, alternative='two-sided')
                nl_m = nl_vals.mean()
                it_m = it_vals.mean()
                direction = '↑' if it_m > nl_m else '↓'
                pct_chg = ((it_m - nl_m) / nl_m * 100) if nl_m != 0 else float('inf')
                sig = '★' if p < 0.05 else '†' if p < 0.10 else ' '
                print(f'    {sig} {metric}: NL={nl_m:.2f} → IT={it_m:.2f} ({direction}{abs(pct_chg):.1f}%) p={p:.4f}')

DONOR-LEVEL BCR METRICS + Mann-Whitney NL→IT

──────────────────────────────────────────────────────────────────────
LIVER — Donor-level BCR
──────────────────────────────────────────────────────────────────────
  NL (6 donors with BCR):
    BCR cells: 194±165
    Clonality: nan±nan
    Singleton: 1.4%
    IgM: 45.9% | IgG: 30.6% | IgA: 13.4%
    Class-switched: 44.0% | Top clone: 3.0
  IT (5 donors with BCR):
    BCR cells: 122±95
    Clonality: nan±nan
    Singleton: 0.9%
    IgM: 44.4% | IgG: 35.9% | IgA: 16.7%
    Class-switched: 52.6% | Top clone: 2.0
  IA (5 donors with BCR):
    BCR cells: 278±206
    Clonality: nan±nan
    Singleton: 2.2%
    IgM: 56.8% | IgG: 24.9% | IgA: 14.0%
    Class-switched: 38.9% | Top clone: 4.0
  AR (1 donors with BCR):
    BCR cells: 59±nan
    Clonality: nan±nan
    Singleton: 0.5%
    IgM: 50.8% | IgG: 22.0% | IgA: 25.4%
    Class-switched: 47.5% | Top clone: 1.0
  CR: no BCR data

  --- Mann-Whitney NL→IT (Liver) ---
      pct_singleton: NL=1.35 →

/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/2756218150.py:26: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp

In [15]:
# Cell 5: DONOR-LEVEL TCR metrics + Mann-Whitney
print(f'{"="*70}')
print('DONOR-LEVEL TCR METRICS + Mann-Whitney NL→IT')
print(f'{"="*70}')

def donor_tcr(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        tcr = grp[grp['TCR_clone.id'].notna()]
        nt = len(tcr)
        if nt == 0:
            rows.append({'Stage':stage,'donor':donor,'n_tcr':0,'pct_tcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,'top_clone':0})
            continue
        cc = tcr['TCR_clone.id'].value_counts()
        nu = len(cc)
        ns = (cc==1).sum()
        if nu > 1:
            fr = cc.values / cc.sum()
            ent = -np.sum(fr * np.log2(fr))
            clon = 1 - ent/np.log2(nu)
        else:
            clon = 0
        rows.append({'Stage':stage,'donor':donor,'n_tcr':nt,
                     'pct_tcr':nt/n*100,'clonality':clon,
                     'pct_singleton':ns/nu*100,'top_clone':cc.max()})
    return pd.DataFrame(rows)

for tissue_val in ['Liver','Blood']:
    df = donor_tcr(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — Donor-level TCR')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[df['Stage']==stage]
        if len(s)==0 or s['n_tcr'].sum()==0:
            print(f'  {stage}: no TCR data')
            continue
        valid = s[s['n_tcr']>0]
        print(f'  {stage} ({len(valid)} donors with TCR):')
        print(f'    TCR cells: {valid.n_tcr.mean():.0f}±{valid.n_tcr.std():.0f}')
        print(f'    Clonality: {valid.clonality.mean():.4f}±{valid.clonality.std():.4f}')
        print(f'    Singleton: {valid.pct_singleton.mean():.1f}%')
        print(f'    Top clone: {valid.top_clone.mean():.0f}')

    nl = df[(df['Stage']=='NL') & (df['n_tcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_tcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  --- Mann-Whitney NL→IT ({tissue_val}) ---')
        for metric in ['clonality','pct_singleton','pct_tcr','top_clone']:
            nl_v = nl[metric].dropna()
            it_v = it[metric].dropna()
            if len(nl_v)>=2 and len(it_v)>=2:
                stat, p = mannwhitneyu(nl_v, it_v, alternative='two-sided')
                nl_m, it_m = nl_v.mean(), it_v.mean()
                d = '↑' if it_m > nl_m else '↓'
                pct = ((it_m-nl_m)/nl_m*100) if nl_m!=0 else float('inf')
                sig = '★' if p<0.05 else '†' if p<0.10 else ' '
                print(f'    {sig} {metric}: NL={nl_m:.3f} → IT={it_m:.3f} ({d}{abs(pct):.1f}%) p={p:.4f}')

DONOR-LEVEL TCR METRICS + Mann-Whitney NL→IT

──────────────────────────────────────────────────────────────────────
LIVER — Donor-level TCR
──────────────────────────────────────────────────────────────────────
  NL (6 donors with TCR):
    TCR cells: 918±890
    Clonality: nan±nan
    Singleton: 0.8%
    Top clone: 135
  IT (6 donors with TCR):
    TCR cells: 1449±1405
    Clonality: nan±nan
    Singleton: 1.6%
    Top clone: 87
  IA (5 donors with TCR):
    TCR cells: 3457±2196
    Clonality: nan±nan
    Singleton: 3.9%
    Top clone: 98
  AR (1 donors with TCR):
    TCR cells: 1767±nan
    Clonality: nan±nan
    Singleton: 1.9%
    Top clone: 151
  CR: no TCR data

  --- Mann-Whitney NL→IT (Liver) ---
      pct_singleton: NL=0.806 → IT=1.590 (↑97.2%) p=0.3939
    † pct_tcr: NL=18.806 → IT=35.594 (↑89.3%) p=0.0649
      top_clone: NL=135.333 → IT=87.000 (↓35.7%) p=0.7483


/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykern


──────────────────────────────────────────────────────────────────────
BLOOD — Donor-level TCR
──────────────────────────────────────────────────────────────────────
  NL (5 donors with TCR):
    TCR cells: 1079±1439
    Clonality: nan±nan
    Singleton: 1.4%
    Top clone: 115
  IT (5 donors with TCR):
    TCR cells: 2023±1473
    Clonality: nan±nan
    Singleton: 4.1%
    Top clone: 57
  IA (4 donors with TCR):
    TCR cells: 3168±790
    Clonality: nan±nan
    Singleton: 6.0%
    Top clone: 81
  AR (1 donors with TCR):
    TCR cells: 2847±nan
    Clonality: nan±nan
    Singleton: 4.7%
    Top clone: 519
  CR: no TCR data

  --- Mann-Whitney NL→IT (Blood) ---
      pct_singleton: NL=1.352 → IT=4.135 (↑205.9%) p=0.1508
      pct_tcr: NL=16.692 → IT=31.840 (↑90.7%) p=0.3095
      top_clone: NL=115.400 → IT=56.800 (↓50.8%) p=0.5296


/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: divide by zero encountered in log2
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykernel_3628/742138273.py:22: RuntimeWarning: invalid value encountered in multiply
  ent = -np.sum(fr * np.log2(fr))
/tmp/ipykern

In [16]:
# Cell 5b: FIX — Safe clonality + rerun all Mann-Whitney
# Paste this AFTER Cell 5 and run it. Overwrites both functions.
from scipy.stats import mannwhitneyu
import numpy as np, pandas as pd

def safe_clonality(clone_counts):
    """Calculate clonality with safe log2 handling."""
    nu = len(clone_counts)        # unique clones
    nt = clone_counts.sum()       # total cells with clone
    if nu <= 0 or nt <= 0:
        return 0.0
    if nu == 1:
        return 1.0 if nt > 1 else 0.0  # all same clone = max clonal
    fr = clone_counts.values / nt
    fr = fr[fr > 0]  # remove zeros
    ent = -np.sum(fr * np.log2(fr))
    max_ent = np.log2(nu)
    return 1 - (ent / max_ent) if max_ent > 0 else 0.0

def donor_bcr_v2(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        bcr = grp[grp['BCR_clone.id'].notna()]
        nb = len(bcr)
        if nb == 0:
            rows.append({'Stage':stage,'donor':donor,'n_bcr':0,'pct_bcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'pct_IgM':np.nan,'pct_IgG':np.nan,'pct_IgA':np.nan,
                         'pct_switched':np.nan,'top_clone_size':0})
            continue
        cc = bcr['BCR_clone.id'].value_counts()
        nu = len(cc)
        ns = (cc==1).sum()
        clon = safe_clonality(cc)
        iso = bcr['BCR_CType'].value_counts()
        it = iso.sum()
        igm = iso.get('IGHM',0)/it*100 if it>0 else np.nan
        igg = iso.get('IGHG',0)/it*100 if it>0 else np.nan
        iga = iso.get('IGHA',0)/it*100 if it>0 else np.nan
        switched = (igg or 0) + (iga or 0)
        rows.append({'Stage':stage,'donor':donor,'n_bcr':nb,
                     'pct_bcr':nb/n*100,'clonality':clon,
                     'pct_singleton':ns/nu*100 if nu>0 else np.nan,
                     'pct_IgM':igm,'pct_IgG':igg,'pct_IgA':iga,
                     'pct_switched':switched,'top_clone_size':cc.max()})
    return pd.DataFrame(rows)

def donor_tcr_v2(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        tcr = grp[grp['TCR_clone.id'].notna()]
        nt = len(tcr)
        if nt == 0:
            rows.append({'Stage':stage,'donor':donor,'n_tcr':0,'pct_tcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,'top_clone':0})
            continue
        cc = tcr['TCR_clone.id'].value_counts()
        nu = len(cc)
        ns = (cc==1).sum()
        clon = safe_clonality(cc)
        rows.append({'Stage':stage,'donor':donor,'n_tcr':nt,
                     'pct_tcr':nt/n*100,'clonality':clon,
                     'pct_singleton':ns/nu*100,'top_clone':cc.max()})
    return pd.DataFrame(rows)

def run_mw(df, metric, stage_a='NL', stage_b='IT'):
    a = df[(df['Stage']==stage_a) & (df['n_bcr']>0 if 'n_bcr' in df.columns else df['n_tcr']>0)][metric].dropna()
    b = df[(df['Stage']==stage_b) & (df['n_bcr']>0 if 'n_bcr' in df.columns else df['n_tcr']>0)][metric].dropna()
    if len(a) < 2 or len(b) < 2:
        return None
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    am, bm = a.mean(), b.mean()
    d = '↑' if bm > am else '↓'
    pct = ((bm-am)/am*100) if am != 0 else float('inf')
    sig = '★' if p<0.05 else '†' if p<0.10 else ' '
    return f'{sig} {metric}: {stage_a}={am:.3f} → {stage_b}={bm:.3f} ({d}{abs(pct):.1f}%) p={p:.4f}'

# ========== RUN ALL ==========
print('='*70)
print('CORRECTED BCR DONOR-LEVEL + Mann-Whitney')
print('='*70)

for tissue_val in ['Liver','Blood']:
    df = donor_bcr_v2(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — BCR')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[(df['Stage']==stage) & (df['n_bcr']>0)]
        if len(s)==0:
            print(f'  {stage}: no BCR data')
            continue
        print(f'  {stage} ({len(s)} donors): BCR={s.n_bcr.mean():.0f}±{s.n_bcr.std():.0f}, '
              f'clonality={s.clonality.mean():.4f}, singleton={s.pct_singleton.mean():.1f}%, '
              f'IgM={s.pct_IgM.mean():.1f}% IgG={s.pct_IgG.mean():.1f}% IgA={s.pct_IgA.mean():.1f}% '
              f'switched={s.pct_switched.mean():.1f}%')

    nl = df[(df['Stage']=='NL') & (df['n_bcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_bcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  Mann-Whitney NL→IT ({tissue_val}):')
        for m in ['clonality','pct_singleton','pct_IgM','pct_IgG','pct_IgA','pct_switched','pct_bcr','top_clone_size']:
            r = run_mw(df, m)
            if r: print(f'    {r}')

print(f'\n{"="*70}')
print('CORRECTED TCR DONOR-LEVEL + Mann-Whitney')
print('='*70)

for tissue_val in ['Liver','Blood']:
    df = donor_tcr_v2(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — TCR')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[(df['Stage']==stage) & (df['n_tcr']>0)]
        if len(s)==0:
            print(f'  {stage}: no TCR data')
            continue
        print(f'  {stage} ({len(s)} donors): TCR={s.n_tcr.mean():.0f}±{s.n_tcr.std():.0f}, '
              f'clonality={s.clonality.mean():.4f}, singleton={s.pct_singleton.mean():.1f}%, '
              f'top_clone={s.top_clone.mean():.0f}')

    nl = df[(df['Stage']=='NL') & (df['n_tcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_tcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  Mann-Whitney NL→IT ({tissue_val}):')
        for m in ['clonality','pct_singleton','pct_tcr','top_clone']:
            # Fix: use n_tcr for TCR filter
            a = nl[m].dropna()
            b = it[m].dropna()
            if len(a)>=2 and len(b)>=2:
                stat, p = mannwhitneyu(a, b, alternative='two-sided')
                am, bm = a.mean(), b.mean()
                d = '↑' if bm>am else '↓'
                pct = ((bm-am)/am*100) if am!=0 else float('inf')
                sig = '★' if p<0.05 else '†' if p<0.10 else ' '
                print(f'    {sig} {m}: NL={am:.3f} → IT={bm:.3f} ({d}{abs(pct):.1f}%) p={p:.4f}')

CORRECTED BCR DONOR-LEVEL + Mann-Whitney

──────────────────────────────────────────────────────────────────────
LIVER — BCR
──────────────────────────────────────────────────────────────────────
  NL (6 donors): BCR=194±165, clonality=0.5051, singleton=1.4%, IgM=45.9% IgG=30.6% IgA=13.4% switched=44.0%
  IT (5 donors): BCR=122±95, clonality=0.5334, singleton=0.9%, IgM=44.4% IgG=35.9% IgA=16.7% switched=52.6%
  IA (5 donors): BCR=278±206, clonality=0.4381, singleton=2.2%, IgM=56.8% IgG=24.9% IgA=14.0% switched=38.9%
  AR (1 donors): BCR=59±nan, clonality=0.5676, singleton=0.5%, IgM=50.8% IgG=22.0% IgA=25.4% switched=47.5%
  CR: no BCR data

  Mann-Whitney NL→IT (Liver):
      clonality: NL=0.505 → IT=0.533 (↑5.6%) p=0.4642
      pct_singleton: NL=1.355 → IT=0.947 (↓30.1%) p=0.4642
      pct_IgM: NL=45.906 → IT=44.391 (↓3.3%) p=0.9307
      pct_IgG: NL=30.561 → IT=35.878 (↑17.4%) p=0.3142
      pct_IgA: NL=13.392 → IT=16.735 (↑25.0%) p=0.3602
      pct_switched: NL=43.953 → IT=52.613 (↑

In [17]:
# Cell 6: BCR × Lineage + V-gene IT vs NL + Isotype shift
print(f'{"="*70}')
print('BCR × LINEAGE  |  V-GENE USAGE  |  ISOTYPE SHIFT')
print(f'{"="*70}')

# BCR by lineage
print('\n--- BCR+ cells by lineage ---')
bcr_lin = obs[obs['BCR_clone.id'].notna()].groupby('major_lineage', observed=True).size()
print(bcr_lin.sort_values(ascending=False).to_string())

# TCR by lineage
print('\n--- TCR+ cells by lineage ---')
tcr_lin = obs[obs['TCR_clone.id'].notna()].groupby('major_lineage', observed=True).size()
print(tcr_lin.sort_values(ascending=False).to_string())

# BCR V-gene top5 by Stage × Tissue
print(f'\n--- BCR V-gene Top5 by Stage × Tissue ---')
for tissue_val in ['Liver','Blood']:
    print(f'\n  {tissue_val}:')
    for stage in ['NL','IT','IA','AR']:
        sub = obs[(obs['Stage']==stage)&(obs['tissue']==tissue_val)&obs['BCR_v_gene'].notna()]
        if len(sub)==0:
            continue
        vg = sub['BCR_v_gene'].value_counts()
        tot = vg.sum()
        top5 = [(g, f'{c/tot*100:.1f}%') for g,c in vg.head(5).items()]
        print(f'    {stage} (n={tot}): {top5}')

# BCR subcluster detail (B and PlasmaB)
print(f'\n--- BCR+ by B/PlasmaB subcluster × Stage ---')
b_plasma = obs[(obs['BCR_clone.id'].notna()) &
               (obs['major_lineage'].isin(['B','PlasmaB']))]
sc_stage = b_plasma.groupby(['gut2021_subcluster_v2','Stage'], observed=True).size().unstack(fill_value=0)
for col_order in [['NL','IT','IA','AR','CR']]:
    present = [c for c in col_order[0] if c in sc_stage.columns]
    sc_stage = sc_stage[present]
print(sc_stage.to_string())

BCR × LINEAGE  |  V-GENE USAGE  |  ISOTYPE SHIFT

--- BCR+ cells by lineage ---
major_lineage
B          11539
PlasmaB     1045
CD4_T         58
CD8_T         57
NK            50
Myeloid       15

--- TCR+ cells by lineage ---
major_lineage
CD8_T      30738
CD4_T      28598
NK          4845
gdT           30
PlasmaB       27
Myeloid       25
B             21

--- BCR V-gene Top5 by Stage × Tissue ---

  Liver:
    NL (n=1164): [('IGHV3-23', '7.9%'), ('IGHV3-7', '6.4%'), ('IGHV3-33', '6.2%'), ('IGHV3-15', '6.2%'), ('IGHV4-39', '5.3%')]
    IT (n=610): [('IGHV3-23', '10.7%'), ('IGHV3-33', '9.7%'), ('IGHV4-59', '6.6%'), ('IGHV4-39', '5.2%'), ('IGHV3-21', '5.1%')]
    IA (n=1389): [('IGHV3-23', '9.6%'), ('IGHV3-33', '8.8%'), ('IGHV4-59', '7.8%'), ('IGHV4-39', '7.4%'), ('IGHV4-34', '4.7%')]
    AR (n=59): [('IGHV1-69D', '8.5%'), ('IGHV4-39', '8.5%'), ('IGHV3-7', '6.8%'), ('IGHV3-74', '6.8%'), ('IGHV3-30', '6.8%')]

  Blood:
    NL (n=1817): [('IGHV3-23', '8.5%'), ('IGHV3-33', '7.4%'), ('IGHV

In [19]:
# Cell 7 FIXED: Save all results (uses v2 functions)
import os

SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
os.makedirs(SAVE_DIR, exist_ok=True)

# Donor-level BCR (v2 = safe clonality)
bcr_liver_df = donor_bcr_v2(obs, 'Liver'); bcr_liver_df['tissue'] = 'Liver'
bcr_blood_df = donor_bcr_v2(obs, 'Blood'); bcr_blood_df['tissue'] = 'Blood'
bcr_all = pd.concat([bcr_liver_df, bcr_blood_df], ignore_index=True)
bcr_all.to_csv(f'{SAVE_DIR}/donor_level_BCR_metrics.csv', index=False)
print(f'Saved: donor_level_BCR_metrics.csv ({len(bcr_all)} rows)')

# Donor-level TCR (v2 = safe clonality)
tcr_liver_df = donor_tcr_v2(obs, 'Liver'); tcr_liver_df['tissue'] = 'Liver'
tcr_blood_df = donor_tcr_v2(obs, 'Blood'); tcr_blood_df['tissue'] = 'Blood'
tcr_all = pd.concat([tcr_liver_df, tcr_blood_df], ignore_index=True)
tcr_all.to_csv(f'{SAVE_DIR}/donor_level_TCR_metrics.csv', index=False)
print(f'Saved: donor_level_TCR_metrics.csv ({len(tcr_all)} rows)')

# BCR subcluster (fix: use gut2021_subcluster_v2 directly)
b_plasma = obs[(obs['BCR_clone.id'].notna()) &
               (obs['major_lineage'].isin(['B','PlasmaB']))]
sc_stage = pd.crosstab(b_plasma['gut2021_subcluster_v2'], b_plasma['Stage'])
col_order = [c for c in ['NL','IT','IA','AR','CR'] if c in sc_stage.columns]
sc_stage = sc_stage[col_order]
sc_stage.to_csv(f'{SAVE_DIR}/BCR_by_subcluster_stage.csv')
print(f'Saved: BCR_by_subcluster_stage.csv')
print(sc_stage.to_string())

# BCR isotype by Stage × Tissue
iso_detail = obs[obs['BCR_CType'].notna()].groupby(
    ['Stage','tissue','BCR_CType'], observed=True
).size().reset_index(name='count')
iso_detail.to_csv(f'{SAVE_DIR}/BCR_isotype_detail.csv', index=False)
print(f'\nSaved: BCR_isotype_detail.csv')

print(f'\n✅ All saved to: {SAVE_DIR}')

Saved: donor_level_BCR_metrics.csv (43 rows)
Saved: donor_level_TCR_metrics.csv (43 rows)
Saved: BCR_by_subcluster_stage.csv
Stage                   NL    IT    IA   AR
gut2021_subcluster_v2                      
B_c01-IGHD             967  1236  1624   84
B_c02-STX16            211   238   546   30
B_c03-CD1C             245  1099   873  231
B_c04-COCH             229   721  1018  180
B_c05-TCL1A            292   123   256   18
B_c06-CD70             184   109   333   29
B_c07-FCRL5            143   243   267   10
plasmaB_c01-SDC1       521    53    74    3
plasmaB_c02-CD52        49    89   101   19
plasmaB_c03-MKI67       81    22    31    2

Saved: BCR_isotype_detail.csv

✅ All saved to: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR
